# Weed species: global → Australian generalisation — DL vs classical ML

**Question.** Do models trained on *globally-sourced* imagery of eight invasive weed species
generalise to *Australian field* conditions? We compare two model families:

* **DL:** ResNet-50 fine-tuned end-to-end on the global set.
* **ML:** Random Forest on hand-crafted colour + texture features (the "traditional" baseline).

**Data.**
* *Train (global):* images of the 8 DeepWeeds species pulled from GBIF/iNaturalist, **excluding Australia**.
* *Test (Australian):* the DeepWeeds dataset (Queensland rangelands).
* *Control:* a held-out **global** test split, so we can separate the geographic gap from plain image noise.

Run this on **Colab Pro** (open network + GPU). It was written to run top-to-bottom; per-species
image counts from GBIF vary, so check the counts printed in Step 1 before training — thin species
can be dropped.

> Note: v1 evaluates the **8 species only** and drops the DeepWeeds *negative* class. Add it back later
> by pulling non-target flora into the global set as a 9th class.


In [1]:
# Step 0 — install (already installed locally into .venv; skipping here)
# pip install pygbif tensorflow tensorflow-datasets scikit-learn scikit-image opencv-python-headless pillow requests


In [2]:
# Step 0 — imports & config
import os, time, io, random, shutil
import numpy as np
import requests
from PIL import Image

import tensorflow as tf
import tensorflow_datasets as tfds
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, roc_auc_score, classification_report, confusion_matrix

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

# Our canonical class order (index -> species). Chosen to match DeepWeeds' species order.
SPECIES = [
    ("Ziziphus mauritiana",      "chinee_apple"),
    ("Lantana camara",           "lantana"),
    ("Parkinsonia aculeata",     "parkinsonia"),
    ("Parthenium hysterophorus", "parthenium"),
    ("Vachellia nilotica",       "prickly_acacia"),   # syn. Acacia nilotica
    ("Cryptostegia grandiflora", "rubber_vine"),
    ("Chromolaena odorata",      "siam_weed"),
    ("Stachytarpheta",           "snake_weed"),        # genus
]
CLASS_NAMES = [c for _, c in SPECIES]
NUM_CLASSES = len(SPECIES)

IMAGES_PER_CLASS = 400        # cap pulled per species (raise later)
EXCLUDE_COUNTRY  = "AU"       # keep training strictly non-Australian
ALLOWED_LICENSES = ["CC0", "CC_BY"]   # add "CC_BY_NC" to boost counts (academic use; not for redistribution)

DL_SIZE  = 224                # ResNet input
FEAT_SIZE = 128               # size for hand-crafted feature extraction

ROOT = os.path.join(os.getcwd(), "weeds")   # local run (was /content/weeds on Colab)
GLOBAL_DIR = os.path.join(ROOT, "global_raw")   # global_raw/<class>/*.jpg
os.makedirs(GLOBAL_DIR, exist_ok=True)


/Users/plabon/projects/Mres/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Step 1 — Build the global training set from GBIF (non-Australian, CC-licensed)

For each species we resolve a taxon key, then page through StillImage occurrences, skip any record
from Australia, keep only permissively licensed media, and download up to `IMAGES_PER_CLASS`.
**Check the printed counts** — species with too few images should be dropped from `SPECIES`.


In [3]:
import glob

from pygbif import species as gbif_species
from pygbif import occurrences as occ

def with_retries(fn, *args, attempts=5, **kwargs):
    for attempt in range(attempts):
        try:
            return fn(*args, **kwargs)
        except Exception:
            if attempt == attempts - 1:
                raise
            time.sleep(5 * (attempt + 1))

def taxon_key(name):
    m = with_retries(gbif_species.name_backbone, scientificName=name)
    usage = m.get("usage") or {}
    key = usage.get("key", m.get("usageKey"))
    return int(key) if key is not None else None

def license_ok(lic):
    # GBIF returns full CC legalcode URLs, e.g.
    # "http://creativecommons.org/licenses/by/4.0/legalcode" or
    # ".../publicdomain/zero/1.0/legalcode" — match those precisely rather
    # than looking for a "CC_BY"-style tag, which never appears verbatim.
    lic = (lic or "").lower()
    if "publicdomain/zero" in lic:
        return "CC0" in ALLOWED_LICENSES
    if "licenses/by-nc/" in lic:
        return "CC_BY_NC" in ALLOWED_LICENSES
    if "licenses/by/" in lic:
        return "CC_BY" in ALLOWED_LICENSES
    return False

def download_species(sci_name, cls_name, target=IMAGES_PER_CLASS):
    cls_dir = os.path.join(GLOBAL_DIR, cls_name)
    os.makedirs(cls_dir, exist_ok=True)
    saved = len(glob.glob(os.path.join(cls_dir, "*.jpg")))
    if saved >= target:            # resume: species already fully downloaded
        return saved
    key = taxon_key(sci_name)
    offset = 0
    while saved < target and offset < 8000:
        try:
            res = with_retries(occ.search, taxonKey=key, mediaType="StillImage", limit=300, offset=offset)
        except Exception as e:
            print(f"  giving up on offset={offset} after retries: {e}")
            break
        recs = res.get("results", [])
        if not recs:
            break
        for r in recs:
            if r.get("countryCode") == EXCLUDE_COUNTRY:
                continue
            if not license_ok(r.get("license")):
                continue
            for m in r.get("media", []):
                url = m.get("identifier")
                if not url:
                    continue
                try:
                    resp = requests.get(url, timeout=15)
                    img = Image.open(io.BytesIO(resp.content)).convert("RGB").resize((DL_SIZE, DL_SIZE))
                    img.save(os.path.join(cls_dir, f"{cls_name}_{saved:04d}.jpg"))
                    saved += 1
                except Exception:
                    continue
                if saved >= target:
                    break
            if saved >= target:
                break
        offset += 300
    return saved

counts = {}
for sci, cls in SPECIES:
    try:
        n = download_species(sci, cls)
    except Exception as e:
        print(f"  {cls}: download failed after retries ({e}); leaving partial for a later resume")
        n = len(glob.glob(os.path.join(GLOBAL_DIR, cls, "*.jpg")))
    counts[cls] = n
    print(f"{cls:16s} {n:4d} images")
print("\nTotal:", sum(counts.values()))


chinee_apple      400 images
lantana           400 images
parkinsonia       400 images
parthenium        400 images
prickly_acacia    400 images
rubber_vine       400 images
siam_weed         400 images
snake_weed        400 images

Total: 3200


In [4]:
# Step 1b — split global images into train / val / test (70/15/15) per class
import glob
GTRAIN, GVAL, GTEST = (os.path.join(ROOT, d) for d in ["global_train", "global_val", "global_test"])
for d in (GTRAIN, GVAL, GTEST):
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d)

for cls in CLASS_NAMES:
    files = sorted(glob.glob(os.path.join(GLOBAL_DIR, cls, "*.jpg")))
    random.shuffle(files)
    n = len(files); n_tr = int(0.70*n); n_va = int(0.15*n)
    parts = {GTRAIN: files[:n_tr], GVAL: files[n_tr:n_tr+n_va], GTEST: files[n_tr+n_va:]}
    for dst, fl in parts.items():
        os.makedirs(os.path.join(dst, cls), exist_ok=True)
        for f in fl:
            shutil.copy(f, os.path.join(dst, cls, os.path.basename(f)))
print("split done")


split done


## Step 2 — DeepWeeds (Australian test set)

Load via TFDS, **verify the class order matches ours**, drop the negative class (label 8),
and materialise arrays for a unified evaluation across both models.


In [5]:
ds_info = tfds.builder("deep_weeds").info
print("DeepWeeds TFDS class order:", ds_info.features["label"].names)
# Expected: ['chinee apple','lantana','parkinsonia','parthenium','prickly acacia',
#            'rubber vine','siam weed','snake weed','negative']
# Our indices 0..7 must line up with the first 8 above. If not, remap here.

ds = tfds.load("deep_weeds", split="train", as_supervised=True)

au_imgs_dl, au_imgs_feat, au_labels = [], [], []
for img, lbl in tfds.as_numpy(ds):
    lbl = int(lbl)
    if lbl == 8:          # drop 'negative' for v1
        continue
    pil = Image.fromarray(img).convert("RGB")
    au_imgs_dl.append(np.array(pil.resize((DL_SIZE, DL_SIZE))))
    au_imgs_feat.append(np.array(pil.resize((FEAT_SIZE, FEAT_SIZE))))
    au_labels.append(lbl)

X_au_dl   = np.array(au_imgs_dl)
X_au_feat = np.array(au_imgs_feat)
y_au      = np.array(au_labels)
print("DeepWeeds (8-class) test set:", X_au_dl.shape, "labels:", np.bincount(y_au))


2026-07-06 13:39:44.778005: W external/local_xla/xla/tsl/platform/cloud/google_auth_provider.cc:185] All attempts to get a Google authentication bearer token failed, returning an empty token. Retrieving token from files failed with "NOT_FOUND: Could not locate the credentials file.". Retrieving token from GCE failed with "FAILED_PRECONDITION: Error executing an HTTP request: libcurl code 6 meaning 'Could not resolve hostname', error details: Could not resolve host: metadata.google.internal".


INFO:Load pre-computed DatasetInfo (eg: splits, num examples,...) from GCS: deep_weeds/3.0.0


INFO:Load dataset info from /var/folders/fc/fsqr_wfd2s37fgl0mxr28clw0000gn/T/tmpf7xm1r_gtfds


INFO:For 'deep_weeds/3.0.0': fields info.[release_notes, splits, supervised_keys, module_name] differ on disk and in the code. Keeping the one from code.


DeepWeeds TFDS class order: ['Chinee apple', 'Lantana', 'Parkinsonia', 'Parthenium', 'Prickly acacia', 'Rubber vine', 'Siam weed', 'Snake weed', 'Negative']


...

INFO:Constructing tf.data.Dataset deep_weeds for split train, from /Users/plabon/tensorflow_datasets/deep_weeds/3.0.0


2026-07-06 13:40:05.717953: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:396] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608


Dataset deep_weeds downloaded and prepared to /Users/plabon/tensorflow_datasets/deep_weeds/3.0.0. Subsequent calls will reuse this data.


2026-07-06 13:40:12.224842: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


DeepWeeds (8-class) test set: (8403, 224, 224, 3) labels: [1125 1064 1031 1022 1062 1009 1074 1016]


## Step 3 — DL model: fine-tuned ResNet-50

In [6]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

def make_ds(folder, training):
    d = tf.keras.utils.image_dataset_from_directory(
        folder, labels="inferred", label_mode="int",
        class_names=CLASS_NAMES, image_size=(DL_SIZE, DL_SIZE),
        batch_size=32, shuffle=training, seed=SEED)
    aug = tf.keras.Sequential([layers.RandomFlip("horizontal"),
                               layers.RandomRotation(0.2),
                               layers.RandomZoom(0.2)])
    def prep(x, y):
        if training: x = aug(x)
        return preprocess_input(tf.cast(x, tf.float32)), y
    return d.map(prep).prefetch(tf.data.AUTOTUNE)

train_ds, val_ds = make_ds(GTRAIN, True), make_ds(GVAL, False)

base = ResNet50(include_top=False, weights="imagenet", pooling="avg",
                input_shape=(DL_SIZE, DL_SIZE, 3))
base.trainable = False
dl_model = models.Sequential([base, layers.Dropout(0.3),
                              layers.Dense(NUM_CLASSES, activation="softmax")])
dl_model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
dl_model.fit(train_ds, validation_data=val_ds, epochs=8)

# fine-tune top of backbone
base.trainable = True
for l in base.layers[:-30]: l.trainable = False
dl_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                 loss="sparse_categorical_crossentropy", metrics=["accuracy"])
dl_model.fit(train_ds, validation_data=val_ds, epochs=6)


Found 2240 files belonging to 8 classes.


Found 480 files belonging to 8 classes.


       0/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0s/step

   49152/94765736 ━━━━━━━━━━━━━━━━━━━━ 6:04 4us/step

   81920/94765736 ━━━━━━━━━━━━━━━━━━━━ 7:11 5us/step

...

67/70 ━━━━━━━━━━━━━━━━━━━━ 2s 754ms/step - accuracy: 0.7921 - loss: 0.5725

68/70 ━━━━━━━━━━━━━━━━━━━━ 1s 754ms/step - accuracy: 0.7921 - loss: 0.5726

69/70 ━━━━━━━━━━━━━━━━━━━━ 0s 754ms/step - accuracy: 0.7920 - loss: 0.5728

70/70 ━━━━━━━━━━━━━━━━━━━━ 0s 754ms/step - accuracy: 0.7919 - loss: 0.5729

70/70 ━━━━━━━━━━━━━━━━━━━━ 61s 868ms/step - accuracy: 0.7918 - loss: 0.5731 - val_accuracy: 0.7312 - val_loss: 0.7623


## Step 4 — ML model: hand-crafted features + Random Forest

In [7]:
import cv2
from skimage.feature import graycomatrix, graycoprops

def extract_features(img):
    # img: HxWx3 uint8 RGB
    feats = []
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    for ch in range(3):
        h = cv2.calcHist([hsv], [ch], None, [16], [0, 256]).flatten()
        feats.extend(h / (h.sum() + 1e-6))
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    q = (gray // 32).astype(np.uint8)                       # 8 grey levels
    glcm = graycomatrix(q, [1], [0], levels=8, symmetric=True, normed=True)
    for p in ["contrast", "homogeneity", "energy", "correlation"]:
        feats.append(float(graycoprops(glcm, p)[0, 0]))
    feats.append(cv2.Canny(gray, 100, 200).mean() / 255.0)  # edge density
    return np.array(feats, dtype=np.float32)

def features_from_folder(folder):
    X, y = [], []
    for idx, cls in enumerate(CLASS_NAMES):
        for f in glob.glob(os.path.join(folder, cls, "*.jpg")):
            img = np.array(Image.open(f).convert("RGB").resize((FEAT_SIZE, FEAT_SIZE)))
            X.append(extract_features(img)); y.append(idx)
    return np.array(X), np.array(y)

Xtr, ytr = features_from_folder(GTRAIN)
ml_model = RandomForestClassifier(n_estimators=400, class_weight="balanced",
                                  random_state=SEED, n_jobs=-1)
ml_model.fit(Xtr, ytr)
print("RF trained on", Xtr.shape, "features")


RF trained on (2240, 53) features


## Step 5 — Evaluate both models: global-test (control) vs DeepWeeds (Australian)

The **gap** = macro-F1(global) − macro-F1(Australian). A large gap for one model and not the other
is the headline result. Per-class F1 tells you *which* species fail to transfer.


In [8]:
def dl_prob(imgs):   # imgs: uint8 RGB arrays at DL_SIZE
    return dl_model.predict(preprocess_input(imgs.astype("float32")), verbose=0)

def ml_prob(feat_imgs):  # uint8 RGB arrays at FEAT_SIZE
    F = np.array([extract_features(im) for im in feat_imgs])
    return ml_model.predict_proba(F)

def report(tag, y_true, y_prob):
    y_pred = y_prob.argmax(1)
    macro = f1_score(y_true, y_pred, average="macro")
    try:
        auroc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
    except Exception:
        auroc = float("nan")
    print(f"\n=== {tag} ===  macro-F1={macro:.3f}  AUROC={auroc:.3f}")
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, zero_division=0))
    return macro

# global-test control (load arrays)
gt_dl, gt_feat, gt_y = [], [], []
for idx, cls in enumerate(CLASS_NAMES):
    for f in glob.glob(os.path.join(GTEST, cls, "*.jpg")):
        pil = Image.open(f).convert("RGB")
        gt_dl.append(np.array(pil.resize((DL_SIZE, DL_SIZE))))
        gt_feat.append(np.array(pil.resize((FEAT_SIZE, FEAT_SIZE))))
        gt_y.append(idx)
gt_dl, gt_feat, gt_y = np.array(gt_dl), np.array(gt_feat), np.array(gt_y)

dl_g = report("DL  | global-test",  gt_y, dl_prob(gt_dl))
dl_a = report("DL  | DeepWeeds/AU", y_au, dl_prob(X_au_dl))
ml_g = report("ML  | global-test",  gt_y, ml_prob(gt_feat))
ml_a = report("ML  | DeepWeeds/AU", y_au, ml_prob(X_au_feat))

print("\nGeneralisation gap (global − AU):")
print(f"  DL: {dl_g - dl_a:+.3f}")
print(f"  ML: {ml_g - ml_a:+.3f}")



=== DL  | global-test ===  macro-F1=0.758  AUROC=0.966
                precision    recall  f1-score   support

  chinee_apple       0.58      0.75      0.66        60
       lantana       0.83      0.92      0.87        60
   parkinsonia       0.76      0.90      0.82        60
    parthenium       0.94      0.77      0.84        60
prickly_acacia       0.64      0.53      0.58        60
   rubber_vine       0.82      0.78      0.80        60
     siam_weed       0.72      0.70      0.71        60
    snake_weed       0.83      0.72      0.77        60

      accuracy                           0.76       480
     macro avg       0.77      0.76      0.76       480
  weighted avg       0.77      0.76      0.76       480




=== DL  | DeepWeeds/AU ===  macro-F1=0.281  AUROC=0.710
                precision    recall  f1-score   support

  chinee_apple       0.28      0.46      0.35      1125
       lantana       0.56      0.19      0.29      1064
   parkinsonia       0.45      0.23      0.31      1031
    parthenium       0.22      0.15      0.18      1022
prickly_acacia       0.34      0.48      0.40      1062
   rubber_vine       0.23      0.21      0.22      1009
     siam_weed       0.37      0.22      0.28      1074
    snake_weed       0.18      0.33      0.23      1016

      accuracy                           0.29      8403
     macro avg       0.33      0.29      0.28      8403
  weighted avg       0.33      0.29      0.28      8403


=== ML  | global-test ===  macro-F1=0.529  AUROC=0.875
                precision    recall  f1-score   support

  chinee_apple       0.48      0.37      0.42        60
       lantana       0.59      0.70      0.64        60
   parkinsonia       0.50      0.50      0.


=== ML  | DeepWeeds/AU ===  macro-F1=0.081  AUROC=0.521
                precision    recall  f1-score   support

  chinee_apple       0.13      0.25      0.17      1125
       lantana       0.32      0.06      0.10      1064
   parkinsonia       0.03      0.00      0.01      1031
    parthenium       0.11      0.05      0.07      1022
prickly_acacia       0.14      0.67      0.24      1062
   rubber_vine       0.07      0.02      0.03      1009
     siam_weed       0.41      0.02      0.04      1074
    snake_weed       0.02      0.00      0.00      1016

      accuracy                           0.14      8403
     macro avg       0.15      0.13      0.08      8403
  weighted avg       0.15      0.14      0.08      8403


Generalisation gap (global − AU):
  DL: +0.477
  ML: +0.449


## Notes & next steps

* **If a species is thin** in Step 1, drop it from `SPECIES` (and re-check the DeepWeeds remap in Step 2)
  or add `CC_BY_NC` to `ALLOWED_LICENSES` to boost counts (fine for academic use, not redistribution).
* **Add the negative class** (v2): pull non-target flora into a 9th global class and stop dropping label 8.
* **Swap the ML features to deep features** (alternative framing): replace `extract_features` with the
  frozen ResNet-50 GAP vector and feed those to the Random Forest / an SVM.
* **VLM arm (later):** add BioCLIP 2 zero-shot + few-shot as a third model on the same DeepWeeds test.
* **XAI probe (later):** Grad-CAM on the DL model, global vs AU images — does attention stay on the plant?
